# Practical 5: CNN for Binary Image Classification (Cats vs. Dogs)


## 1. Dataset Preparation


In [ ]:
# Core imports
import os
import numpy as np
import matplotlib.pyplot as plt
import pathlib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Download and extract the Cats vs Dogs (filtered) dataset
DATASET_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"

zip_path = keras.utils.get_file(
    "cats_and_dogs_filtered.zip", origin=DATASET_URL, extract=True
)
base_dir = pathlib.Path(zip_path).parent / "cats_and_dogs_filtered"

train_dir = base_dir / "train"
val_dir = base_dir / "validation"

print("Base dir:", base_dir)
print("Train dir contents:", os.listdir(train_dir))
print("Validation dir contents:", os.listdir(val_dir))


In [ ]:
# Explore the dataset: count images per class
for split_name, split_dir in [("train", train_dir), ("validation", val_dir)]:
    for cls in ["cats", "dogs"]:
        n = len(os.listdir(split_dir / cls))
        print(f"{split_name}/{cls}: {n} images")


In [ ]:
# Visualize a few sample images from each class
def show_samples(directory, cls, n=4):
    files = sorted(os.listdir(directory / cls))[:n]
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, fname in zip(axes, files):
        img = keras.utils.load_img(directory / cls / fname)
        ax.imshow(img)
        ax.set_title(cls)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_dir, "cats")
show_samples(train_dir, "dogs")


In [ ]:
# Image preprocessing: build tf.data datasets directly from the directory structure
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="binary",     # 0 = cats, 1 = dogs
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    val_dir,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

class_names = train_ds.class_names
print("Class names (0/1):", class_names)


In [ ]:
# Rescale pixel values from [0, 255] to [0, 1], and configure the pipeline for performance
normalization_layer = layers.Rescaling(1./255)

AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, augment_layer=None):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    if augment_layer is not None:
        ds = ds.map(lambda x, y: (augment_layer(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

# Data augmentation pipeline (applied only to the training set, only during training)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="data_augmentation")

train_ds_aug = prepare(train_ds, augment_layer=data_augmentation)
train_ds_plain = prepare(train_ds, augment_layer=None)   # used for the "no augmentation" comparison
val_ds_ready = prepare(val_ds, augment_layer=None)        # never augment validation/test data


## 2. CNN Model Development


In [ ]:
def build_cnn():
    model = models.Sequential([
        layers.Input(shape=(150, 150, 3)),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(512, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

cnn_model = build_cnn()
cnn_model.summary()


## 3. Model Training


In [ ]:
EPOCHS = 20

print("=== Training CNN WITH data augmentation ===")
tf.random.set_seed(SEED)
model_augmented = build_cnn()
history_augmented = model_augmented.fit(
    train_ds_aug,
    validation_data=val_ds_ready,
    epochs=EPOCHS,
    verbose=1,
)


In [ ]:
print("=== Training CNN WITHOUT data augmentation (for comparison) ===")
tf.random.set_seed(SEED)
model_plain = build_cnn()
history_plain = model_plain.fit(
    train_ds_plain,
    validation_data=val_ds_ready,
    epochs=EPOCHS,
    verbose=1,
)


In [ ]:
def plot_history(history, title):
    hist = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist["accuracy"], label="Train Accuracy")
    axes[0].plot(hist["val_accuracy"], label="Validation Accuracy")
    axes[0].set_title(f"{title}\nAccuracy vs. Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(hist["loss"], label="Train Loss")
    axes[1].plot(hist["val_loss"], label="Validation Loss")
    axes[1].set_title(f"{title}\nLoss vs. Epoch")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_augmented, "CNN WITH Augmentation")
plot_history(history_plain, "CNN WITHOUT Augmentation")


## 4. Model Evaluation


In [ ]:
# Gather true labels and predictions from the validation set
y_true, y_pred_prob = [], []
for images, labels in val_ds_ready:
    preds = model_augmented.predict(images, verbose=0)
    y_pred_prob.extend(preds.flatten())
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true).astype(int)
y_pred_prob = np.array(y_pred_prob)
y_pred = (y_pred_prob >= 0.5).astype(int)   # sigmoid threshold at 0.5

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# Confusion matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - CNN (with augmentation)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side accuracy comparison: with vs. without augmentation
final_acc_aug = history_augmented.history["val_accuracy"][-1]
final_acc_plain = history_plain.history["val_accuracy"][-1]

print(f"Final validation accuracy WITH augmentation:    {final_acc_aug:.4f}")
print(f"Final validation accuracy WITHOUT augmentation: {final_acc_plain:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(history_augmented.history["val_accuracy"], label="With Augmentation")
plt.plot(history_plain.history["val_accuracy"], label="Without Augmentation")
plt.title("Validation Accuracy: Augmentation vs. No Augmentation")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Interpreting Predictions on Individual Images


In [ ]:
# Grab one batch of validation images (unnormalized, for display) and predict
val_ds_display = keras.utils.image_dataset_from_directory(
    val_dir, labels="inferred", label_mode="binary",
    image_size=IMG_SIZE, batch_size=16, shuffle=True, seed=SEED,
)

images_batch, labels_batch = next(iter(val_ds_display))
images_norm = normalization_layer(images_batch)
preds_batch = model_augmented.predict(images_norm, verbose=0).flatten()

plt.figure(figsize=(16, 9))
for i in range(min(12, images_batch.shape[0])):
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(images_batch[i].numpy().astype("uint8"))
    true_label = class_names[int(labels_batch[i].numpy())]
    pred_label = class_names[int(preds_batch[i] >= 0.5)]
    confidence = preds_batch[i] if preds_batch[i] >= 0.5 else 1 - preds_batch[i]
    color = "green" if true_label == pred_label else "red"
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence:.2f})", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()
